# Importing Libraries

In [1]:
from umap import UMAP # for UMAP latent space projections
import sys # for relative imports of sigma

Using relative imports for sigma2 and its packages

In [2]:
sys.path.insert(0,"..")

from sigma.utils import normalisation as norm 
from sigma.utils import visualisation as visual
from sigma.utils.load import SEMDataset
from sigma.src.utils import same_seeds
from sigma.src.dim_reduction import Experiment
from sigma.models.autoencoder import AutoEncoder
from sigma.src.segmentation import PixelSegmenter
from sigma.gui import gui
from sigma.utils.loadtem import TEMDataset

from sigma.utils import batch_proc as batch

# Loading the data

Testing projection and clustering parameters for a single tile in the batch




In [3]:
file_path='batch_test/Project 1 Specimen 1 Area 1 Site 1 Map Data 3.h5oina'
sem=SEMDataset(file_path)

Resizing nav_img from (1408, 2048) to (352, 512)
No X-ray line metadata found — defaulting to empty


# Viewing the data

The dataset can be viewed with `gui.view_dataset`. In the gui, it is possilbe to view the summed spectra across the dataset, search for edges near a particular energy, and define edges of interest in the feature list.

Elemental maps for these features are shown in the 'Elemental maps' tab.

In [7]:
sem.set_feature_list(['O_Ka','C_Ka','Fe_Ka','Si_Ka','S_Ka','N_Ka'])

Set feature_list to ['O_Ka', 'C_Ka', 'Fe_Ka', 'Si_Ka', 'S_Ka', 'N_Ka']


In [8]:
gui.view_dataset(sem)

Output()

Output()

# Processing steps before dimensionality reduction

To improve performance, it is often desireable to bin the signal in the navigation dimension.
This is done with `sem.rebin_signal(size=(nx,ny))`, which will bin the signal by nx,ny

The signals for each pixel are also normalised with `sem.peak_intensity_normalisation`

The initial 0 energy peak is removed from all signals with `sem.remove_first_peak(end=float)` where `end` specifies the energy at which the first peak ends

In [9]:
# Rebin both edx and bse dataset
sem.rebin_signal(size=(3,3))

# normalisation to make the spectrum of each pixel summing to 1.
sem.peak_intensity_normalisation()

# Remove the first peak until the energy of 0.1 keV

sem.remove_first_peak(end=0.1)


Rebinning the intensity with the size of (3, 3)
Normalising the chemical intensity along axis=2, so that the sum is equal to 1 along axis=2.
Removing the first peak by setting the intensity to zero until the energy of 0.1 keV.


In [10]:
#sem.normalisation([norm.neighbour_averaging])
sem.normalisation([norm.neighbour_averaging,norm.zscore])
#sem.normalisation([norm.neighbour_averaging,norm.zscore,norm.softmax])

Set feature_list to ['O_Ka', 'C_Ka', 'Fe_Ka', 'Si_Ka', 'S_Ka', 'N_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore


In [11]:
gui.view_pixel_distributions(sem,norm_list=[norm.neighbour_averaging,norm.zscore,norm.softmax],cmap='Reds')

In [12]:
print('After normalisation:')
gui.view_intensity_maps(spectra=sem.normalised_elemental_data, element_list=sem.feature_list)

After normalisation:


## Adding electron intensity as a component for projection

If you wish to add the BSE image (or other navigation image) to the feature vectors that are used for clustering, you can do so with 
`sem.get_feature_maps_with_nav_img()`

You will need to specify the normalisation steps you performed on the eds data on this image, as shown below

The bse image will be automatically resized to be the same dimensions as the eds maps


In [13]:
sem.get_feature_maps_with_nav_img(normalisation=[norm.neighbour_averaging,norm.zscore])

Resizing nav_img from (352, 512) to (117, 170)


In [14]:
gui.view_dataset(sem)

Output()

Output()

# Dimensionality Reduction

The above steps create an $x \times y \times n$ data cube. To perform clustering, it is necessary to project this into a lower dimensonal latent space

Dimensionality reduction may be performed either with the [UMAP algorithm](https://arxiv.org/abs/1802.03426) or with an autoencoder (more details on the autoencoder can be found [here](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2022GC010530) )

It is not necessary to run both dimensionality reductions, but both are demonstrated in this notebook for completeness.

## Latent Space Projection with UMAP

In [15]:
data = sem.normalised_elemental_data.reshape(-1,len(sem.feature_list))
umap = UMAP(
        n_neighbors=15,
        min_dist=0.02,
        n_components=2,
        metric='euclidean'
    )
latent = umap.fit_transform(data)

In [6]:
gui.show_projection(latent)

NameError: name 'latent' is not defined

# Pixel Segmentation

Once a latent space is formed, it can be split into clusters using segmentation algorithms.

The first one demsonstated is Gaussian Mixture Modelling (GMM)

## Segmentation with GMM

First, create a `PixelSegmenter` object 

In [17]:
ps_gmm=PixelSegmenter(latent=latent,
                      dataset=sem,method='GaussianMixture',
                      method_args={'n_components' :50,
                                   'random_state':0, 'init_params':'kmeans'} )

In [5]:
gui.check_latent_space(ps=ps_gmm,show_map=True,ratio_to_be_shown=1.0,alpha_cluster_map=0.5)


KeyboardInterrupt



In [3]:
batch_results = batch.batch_process_and_extract(inputs='batch_test/',
                          preprocessing_args={'rebin':True,'bin_size':(3,3),
                                              'peak_int_norm':True,
                                              'remove_first_peak':True,
                                              'first_peak_end':0.1,
                                              'norm_methods':('neighbour_averaging','zscore','softmax'),
                                              'use_nav_img':True,
                                              'xray_lines':['O_Ka','C_Ka','Fe_Ka','Si_Ka','S_Ka','N_Ka']},
                          projection='umap',
                          projection_args={'n_neighbours':15,'min_dist':0.01,'n_components':2,
                                           'metric':'euclidean'},
                          clustering='GaussianMixture',
                          clustering_args={'n_components':50,
                                           'random_state':0,
                                           'init_params':'kmeans'},
                          extractor_kwargs={'normalised':True,
                                            'return_hyperspy':False}
                         )

Resizing nav_img from (1408, 2048) to (352, 512)
No X-ray line metadata found — defaulting to empty

➡️ Processing 'tutorials/real_batch_test/Batch Data for Tom/Project 1 Specimen 1 Area 1 Site 1 Map Data 3.h5oina'
Set feature_list to ['O_Ka', 'C_Ka', 'Fe_Ka', 'Si_Ka', 'S_Ka', 'N_Ka']
  - Rebinning: (3, 3)
Rebinning the intensity with the size of (3, 3)
  - Peak intensity normalisation
Normalising the chemical intensity along axis=2, so that the sum is equal to 1 along axis=2.
  - Removing first peak up to 0.1
Removing the first peak by setting the intensity to zero until the energy of 0.1 keV.
  - Normalising with: ('neighbour_averaging', 'zscore', 'softmax')
Set feature_list to ['O_Ka', 'C_Ka', 'Fe_Ka', 'Si_Ka', 'S_Ka', 'N_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore
    3. softmax
Resizing nav_img from (352, 512) to (117, 170)
File 0: Pre-processing complete
performing umap projection on file 0
Performing Clustering
Clustering Successful
labels shape: (1989

In [4]:
nmf_result = batch.nmf_on_batch_cluster_spectra(
    batch_results,
    n_components=5,
    normalised=True,
    method_args={"init": "nndsvda", "random_state": 0, "max_iter": 1000},
)

In [5]:
batch.show_batch_unmixed_weights_and_components(
    nmf_result,
    peak_list=["O_Ka", "Fe_Ka", "Si_Ka","S_Ka","Na_Ka","Ni_Ka","Fe_Kb","Fe_La","Al_Ka"]
)

In [6]:
batch.show_batch_abundance_map(batch_results, nmf_result)

In [4]:
batch.show_batch_abundance_map_interactive_tight(batch_results, nmf_result,grid_shape=(8, 8),dpi=300)


KeyboardInterrupt

